# IFD-Fintech — Google Colab Runner

**Robust Federated Learning for Credit Card Identity Fraud Detection**  
*Gated cascade defense framework against adaptive poisoning attacks.*

Three modes: smoke test, full experiment, or FL simulation with real training.

## 1. Setup

Clones repo and installs dependencies.

In [ ]:
import sys, os, json
from pathlib import Path

GIT_REPO = "https://github.com/your-org/IFD-Fintech.git"  # ← set your repo
PROJECT_DIR = "/content/IFD-Fintech"

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 "{GIT_REPO}" "{PROJECT_DIR}"
%cd "{PROJECT_DIR}"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Install all deps (Colab has Python 3.10, ignore version pin)
!pip install -q numpy scikit-learn torch pandas flwr

print(f"\n✓ Setup complete — Python {sys.version.split()[0]}")

## 2. Choose Mode

Click a button below, or use the manual cell.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

mode_info = widgets.HTML(value="<i>Click a button to start.</i>")

def _run_simple(_):
    clear_output(wait=True)
    display(title_html, mode_info)
    mode_info.value = "⏳ Running simple smoke test..."
    !cd "{PROJECT_DIR}" && python3 -m ifd_fintech.test.run_simple
    mode_info.value = "✅ Done — scroll up for results."

def _run_full(_):
    clear_output(wait=True)
    display(title_html, mode_info)
    mode_info.value = "⏳ Running full experiment..."
    !cd "{PROJECT_DIR}" && python3 -m ifd_fintech.test.run_full --attacks all --output /tmp/full.json
    if os.path.exists("/tmp/full.json"):
        with open("/tmp/full.json") as f:
            d = json.load(f)
        mode_info.value = f"✅ Done — {d['total_experiments']} experiments."

def _run_sim(_):
    clear_output(wait=True)
    display(title_html, mode_info)
    mode_info.value = "⏳ Running FL simulation (synthetic data)..."
    from ifd_fintech.experiment.simulation import Simulation
    sim = Simulation(n_clients=10, n_rounds=30, clients_per_round=5)
    sim.load_data(source="synthetic", n_samples=2000, n_features=16)
    sim.build_defense()
    sim.run()
    sim.report()
    mode_info.value = "✅ Simulation finished."

title_html = widgets.HTML("<h3>Select a mode:</h3>")
b1 = widgets.Button(description="▶ Smoke Test", button_style="success",
                    layout=widgets.Layout(width="220px"))
b2 = widgets.Button(description="▶ Full Experiment", button_style="primary",
                    layout=widgets.Layout(width="220px"))
b3 = widgets.Button(description="▶ FL Simulation", button_style="info",
                    layout=widgets.Layout(width="220px"))

b1.on_click(_run_simple)
b2.on_click(_run_full)
b3.on_click(_run_sim)

display(title_html, widgets.HBox([b1, b2, b3]), mode_info)

## Manual Mode

Set `MODE` below and run.

In [ ]:
MODE = "simulation"  # one of: simple, full, simulation

if MODE == "simple":
    !cd "{PROJECT_DIR}" && python3 -m ifd_fintech.test.run_simple
elif MODE == "full":
    !cd "{PROJECT_DIR}" && python3 -m ifd_fintech.test.run_full --attacks all --output /tmp/full.json
elif MODE == "simulation":
    from ifd_fintech.experiment.simulation import Simulation
    sim = Simulation(n_clients=10, n_rounds=30, clients_per_round=5)
    sim.load_data(source="synthetic", n_samples=2000, n_features=16)
    sim.build_defense()
    sim.run()
    sim.report()
else:
    print(f"Unknown MODE: {MODE}")